6. Случайная величина имеет распределение Парето:

$$
p(x) =
\begin{cases} 
\frac{\theta - 1}{x^\theta}, & x \geq 1 \\ 
0, & x < 1 
\end{cases}, \quad \theta > 1.
$$

a) По выборке объема $n$ найти оценку параметра $\theta$ методом максимального правдоподобия.

b) Построить доверительный интервал для медианы.

c) Построить асимптотический доверительный интервал для параметра $\theta$.

d) Сгенерируйте выборку объема $n = 100$ для некоторого значения параметра $\theta$. Вычислите указанные выше доверительные интервалы для доверительной вероятности 0.95.

e) Численно постройте бутстраповский доверительный интервал двумя способами, используя параметрический бутстрап и непараметрический бутстрап.

f) Сравнить все интервалы.

In [20]:
#подключение всех необходимых библиотек

import numpy as np
from scipy import stats

In [21]:
#Генерация выборки

def generate_sample(theta, n):
    y = np.random.uniform(0, 1, n)
    x = (1 - y) ** (-1 / (theta - 1))
    return x
n = 100
theta = 50
sample = generate_sample(theta, n)

In [22]:
#Вычисление интервала для медианы
def calc_for_mean(sample, n):
    ln_mean = np.mean([np.log(x) for x in sample])
    #из задачи получили
    left = 2 ** ln_mean - (1.96 * np.log(2) * 2 ** ln_mean * ln_mean) / np.sqrt(n)
    right = 2 ** ln_mean + (1.96 * np.log(2) * 2 ** ln_mean * ln_mean) / np.sqrt(n)
    return left, right, right-left


In [23]:
l_med, r_med, leng_med = calc_for_mean(sample, n)
print(f"Median exists between {l_med} and {r_med} (l = {l_med}), according to the asymptotic interval")

Median exists between 1.010716375522505 and 1.0159845331848825 (l = 1.010716375522505), according to the asymptotic interval


In [24]:
def calc_omp(sample, n):
    ln_mean = np.mean([np.log(x) for x in sample])
    #получили границы интервала из того, что написано в pdf
    left = 1 - (1.96 - np.sqrt(n)) / (ln_mean * np.sqrt(n))
    right = 1 + (1.96 + np.sqrt(n)) / (ln_mean * np.sqrt(n))
    return left, right, right-left

In [25]:
l_thet, r_thet, leng_thet = calc_omp(sample, n)
print(f"{l_thet} < θ < {r_thet} according to the asymptotic interval (l = {leng_thet})")

43.0212028224429 < θ < 63.50915245726583 according to the asymptotic interval (l = 20.487949634822925)


In [26]:
#bootstrap

def calc_bootstrap_nonparam(sample, n):
    bsp_iter = 1000
    ln_mean = np.mean([np.log(x) for x in sample])
    theta_OMP = 1 + 1 / ln_mean.astype(float)
    bsp_delta = []
    for _ in range(bsp_iter):
        bsp_delta.append(1 + 1 / np.mean([np.log(x) for x in np.random.choice(sample, size=n)]).astype(float) - theta_OMP)
    left = theta_OMP - np.sort(bsp_delta)[974]
    right = theta_OMP - np.sort(bsp_delta)[24]
    return left, right, right - left

def calc_bootstrap_param(sample, n):
    bsp_iter = 50000
    ln_mean = np.mean([np.log(x) for x in sample])
    theta_OMP = 1 + 1 / ln_mean.astype(float)
    bsp_delta = []
    for _ in range(bsp_iter):
        random_sample = []
        for _ in range(n):
            y = np.random.random()
            random_sample.append((1-y)**(1/(1-theta_OMP)))
        bsp_delta.append(1 + 1/np.mean([np.log(x) for x in random_sample]) - theta_OMP)
    left = theta_OMP - sorted(bsp_delta)[int((1 + 0.95) * bsp_iter / 2)]
    right = theta_OMP - sorted(bsp_delta)[int((1 - 0.95) * bsp_iter / 2)]
    return left, right, right - left


In [27]:
l_bsp1, r_bsp1, leng_bsp1 = calc_bootstrap_nonparam(sample, n)
l_bsp2, r_bsp2, leng_bsp2 = calc_bootstrap_param(sample, n)

print(f"Param. bootstrap interval: {l_bsp2} < θ < {r_bsp2} (l = {leng_bsp2})")
print(f"Non-parametric bootstrap interval: {l_bsp1} < θ < {r_bsp1} (l = {leng_bsp1})")

Param. bootstrap interval: 41.164533719240396 < θ < 62.049286165127555 (l = 20.88475244588716)
Non-parametric bootstrap interval: 41.82191736398691 < θ < 61.813236021692525 (l = 19.991318657705612)


In [28]:
intervals = {"bootstrap (non-parametric)" : leng_bsp1, "bootstrap (parametric)": leng_bsp2, "asymptotic": leng_thet}
intervals = sorted(intervals.items(), key=lambda item: item[1])
for interval_name, interval_value in intervals:
    print(f"{interval_name} (l = {interval_value:.2f})\n")

bootstrap (non-parametric) (l = 19.99)

asymptotic (l = 20.49)

bootstrap (parametric) (l = 20.88)

